In [1]:
import pickle
import networkx as nx
import torch 
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings

from typing import Tuple
from tqdm import tqdm

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils.graph_utils import load_graph, save_graph

/home/xyu/.conda/envs/firegnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Learning with Expression Data, Disease KG & Healthy Aging KG

1. Generate sample_2_kg network (sample_ad_kg, sample_healthy_kg as comparison)
   + logFC
   + ecdf
   + z-score
   + average
   + all
2. Networkx to HeteroData
3. Node classification

### 1. Sample Scoring

In [ ]:
df_adni = pd.read_csv("./data/ADNI/adni_gene_cleaned.csv", index_col=0)
adni_design = pd.read_csv("./data/ADNI/design_with_real_target.tsv", sep='\t', index_col=0)
adni_design['Target'] = adni_design['Target'].map({'Control':0, 'AD':1, 'MCI':2})

3 classes ADNI data: Control, AD, MCI

In [24]:
adni_design.to_csv("./data/ADNI/design_3cls.csv")
adni_labels_3cls = adni_design['Target'].to_list()
df_adni_3cls = df_adni.T
df_adni_3cls.to_csv("./data/ADNI/adni_exp_3cls.csv")
df_adni_3cls

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,3.651,2.2865,3.039,2.3395,2.783,2.25300,2.081,7.043,3.30950,2.202,...,5.748,3.77150,4.0365,4.353000,5.1763,1.9045,5.31750,9.2190,7.3010,5.7490
037_S_4410,3.183,2.1230,3.543,2.2085,2.383,2.37550,1.733,6.773,3.27625,2.317,...,5.974,4.17300,4.4415,4.520667,5.0964,1.9265,5.38800,8.3785,6.7580,6.0935
006_S_4153,3.278,2.3545,3.528,2.1745,2.593,2.46825,1.841,6.910,3.20875,2.540,...,5.119,3.91275,4.6210,4.230667,5.1143,2.2315,5.62800,9.1085,7.3365,5.2615
116_S_1232,3.371,2.3725,3.835,2.1545,2.570,2.51925,2.249,7.209,3.24950,2.559,...,4.904,3.73800,4.4435,4.050667,5.1520,2.0545,5.46300,9.3210,7.1685,4.7340
099_S_4205,3.358,2.3865,3.392,2.1720,2.660,2.39725,1.893,6.920,3.15425,2.347,...,5.533,3.94600,4.5215,4.639667,5.2103,2.0405,5.64750,9.0300,7.2025,5.4575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
009_S_2381,3.302,2.5075,3.524,2.2525,2.876,2.76275,2.089,6.805,3.09150,2.650,...,4.523,3.50150,4.4685,3.962333,5.0892,1.9320,5.44275,9.2900,6.7035,4.7915
053_S_4557,3.403,2.3090,3.515,2.3225,3.106,2.85125,2.102,7.265,3.27575,2.603,...,5.087,3.58325,4.1555,4.125667,5.0986,1.9445,5.14000,9.8090,7.2810,4.7055
073_S_4300,3.530,2.4155,3.651,2.0760,2.707,2.38325,2.092,7.375,3.26950,2.557,...,4.938,3.63450,4.5165,4.070333,5.1731,2.0795,5.38775,9.5215,6.9825,5.0785
041_S_4014,3.532,2.4545,3.609,2.3495,3.081,2.69125,2.024,7.257,3.11425,2.497,...,4.037,3.68525,4.0480,3.976333,4.8241,2.0075,5.05725,9.5810,6.6865,3.8660


2 classes ADNI data: Control, AD(Disease)

In [25]:
adni_design_2cls = adni_design[adni_design['Target'] != 2]
# save 
adni_design_2cls.to_csv("./data/ADNI/design_2cls.csv")

adni_labels_2cls = adni_design['Target'].to_list()
df_adni_2cls = df_adni[adni_design_2cls.index.to_list()].T
df_adni_2cls.to_csv("./data/ADNI/adni_exp_2cls.csv")
df_adni_2cls

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,3.651,2.2865,3.039,2.3395,2.783,2.25300,2.081,7.043,3.30950,2.202,...,5.748,3.77150,4.0365,4.353000,5.1763,1.9045,5.31750,9.2190,7.3010,5.7490
037_S_4410,3.183,2.1230,3.543,2.2085,2.383,2.37550,1.733,6.773,3.27625,2.317,...,5.974,4.17300,4.4415,4.520667,5.0964,1.9265,5.38800,8.3785,6.7580,6.0935
006_S_4153,3.278,2.3545,3.528,2.1745,2.593,2.46825,1.841,6.910,3.20875,2.540,...,5.119,3.91275,4.6210,4.230667,5.1143,2.2315,5.62800,9.1085,7.3365,5.2615
116_S_1232,3.371,2.3725,3.835,2.1545,2.570,2.51925,2.249,7.209,3.24950,2.559,...,4.904,3.73800,4.4435,4.050667,5.1520,2.0545,5.46300,9.3210,7.1685,4.7340
128_S_0205,3.194,2.3560,3.146,2.1230,2.673,2.52150,1.766,7.079,3.10525,2.448,...,5.571,4.07650,4.4430,4.366667,5.2429,1.9105,5.56125,8.5345,6.9175,5.6455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
014_S_4668,3.330,2.2820,3.497,2.2965,3.155,2.67350,2.158,6.950,3.33150,2.923,...,4.309,3.60500,4.2075,3.903667,4.9955,2.1810,5.41125,9.3105,6.7660,5.0440
130_S_0289,3.368,2.2950,3.128,1.9650,2.622,2.45225,1.953,6.878,3.10100,2.481,...,5.649,3.86475,4.6765,4.663333,5.1113,2.0300,5.36250,9.1965,7.1770,5.5510
009_S_2381,3.302,2.5075,3.524,2.2525,2.876,2.76275,2.089,6.805,3.09150,2.650,...,4.523,3.50150,4.4685,3.962333,5.0892,1.9320,5.44275,9.2900,6.7035,4.7915
041_S_4014,3.532,2.4545,3.609,2.3495,3.081,2.69125,2.024,7.257,3.11425,2.497,...,4.037,3.68525,4.0480,3.976333,4.8241,2.0075,5.05725,9.5810,6.6865,3.8660


geo dataset

In [26]:
df_geo = pd.read_csv("./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv", index_col=0).T
df_geo

,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDB,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981
GSM1423780,-0.066694,0.023919,-0.031845,0.015204,0.089450,-0.167457,0.023215,0.004379,0.041729,0.074247,...,0.066588,-0.020308,0.080426,-0.057932,0.007106,0.106203,0.063151,0.066667,0.060402,0.023945
GSM1423781,0.054097,-0.012131,-0.030740,0.034282,-0.006780,0.068906,-0.004402,-0.029354,0.003009,-0.035774,...,0.035217,-0.000390,-0.027666,0.033120,0.065091,-0.040029,-0.118265,0.096145,-0.007896,0.022231
GSM1423782,0.025282,0.001929,-0.085577,-0.023002,-0.314342,0.092161,-0.043387,0.069851,0.010428,0.116397,...,0.072900,-0.042220,0.085496,-0.020298,0.040479,-0.146334,-0.027519,0.165024,-0.008110,0.118489
GSM1423783,0.002485,0.034625,-0.066813,-0.018289,0.094836,0.073653,0.073933,-0.043867,0.045298,-0.087155,...,-0.064955,-0.034116,-0.109619,0.029910,0.079794,-0.000814,-0.250529,-0.014006,-0.238408,-0.042106
GSM1423784,-0.092606,0.019591,-0.030884,0.020118,-0.206281,0.050492,-0.003392,0.055446,0.111875,0.152810,...,-0.091836,-0.050365,0.105040,0.052845,-0.013048,0.023013,0.051360,0.054292,0.115933,0.035485
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1424242,-0.035529,-0.067985,-0.115346,-0.045191,-0.085769,-0.041509,-0.018460,0.108033,0.010502,0.153964,...,-0.017694,-0.130394,0.012196,-0.022353,0.032304,-0.100798,0.016300,0.053928,-0.060836,0.176064
GSM1424243,0.075622,-0.010792,-0.088835,-0.021300,-0.096128,-0.037948,0.038506,0.107594,0.001454,0.062382,...,0.025705,-0.133761,-0.092569,0.121448,0.159210,0.017417,-0.108027,0.072586,0.090821,0.010657
GSM1424244,0.065244,-0.087919,-0.120331,-0.038014,0.068436,-0.003720,0.013911,0.034820,0.105181,0.085700,...,-0.043359,-0.048832,-0.040329,0.001067,0.012171,0.130757,0.016830,0.094532,-0.294262,0.086503
GSM1424245,0.054953,0.072740,-0.094495,-0.028996,-0.039163,-0.066074,0.028088,0.070063,-0.028701,0.052751,...,-0.040056,-0.091335,-0.080240,0.045302,0.084873,0.083839,-0.073553,0.106768,-0.118929,0.061755


In [30]:
geo_design = pd.read_csv("./data/GEO/GSE33000_ad_hd/GSE33000_meta_2cls.csv", index_col=0)
geo_design['Target'] = geo_design['Target'].map({'Disease':1, 'Control':0})
geo_labels = geo_design['Target'].to_list()
geo_design

,Tissue,Sample type,Age,Gender,Disease status,Target
GSM1423780,prefrontal cortex brain,reference,67 yrs,female,Alzheimer's disease,1
GSM1423781,prefrontal cortex brain,reference,88 yrs,male,Alzheimer's disease,1
GSM1423782,prefrontal cortex brain,reference,62 yrs,male,Alzheimer's disease,1
GSM1423783,prefrontal cortex brain,reference,90 yrs,female,Alzheimer's disease,1
GSM1423784,prefrontal cortex brain,reference,90 yrs,female,Alzheimer's disease,1
...,...,...,...,...,...,...
GSM1424242,prefrontal cortex brain,reference,52 yrs,male,non-demented,0
GSM1424243,prefrontal cortex brain,reference,61 yrs,male,non-demented,0
GSM1424244,prefrontal cortex brain,reference,60 yrs,female,non-demented,0
GSM1424245,prefrontal cortex brain,reference,55 yrs,female,non-demented,0


In [155]:
def process_and_save(
    data: pd.DataFrame, 
    design: pd.DataFrame, 
    threshold: float, 
    control: str|int,
    do_function,
    output_dir: str,
    method: str = 'logfc',
    **kwards
):
    """
    Wrapper to run the search, show progress, and save results.
    """
    # Create directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    print(f"Starting {method} analysis...")
    
    # Simple progress bar for the high-level step
    with tqdm(total=2, desc="Overall Progress") as pbar:
        # 1. do sample scoring
        output_df, summary_df = do_function(data, design, threshold, control=control, **kwards)
        
        pbar.update(1)
        
        # 2. Save the files
        out_path = os.path.join(output_dir, f"sample_scoring_{method}.csv")
        sum_path = os.path.join(output_dir, f"scoring_summary_{method}.csv")
        #sum_path_2 = os.path.join(output_dir, f"scoring_summary1_{method}.csv")
        
        output_df.to_csv(out_path)
        summary_df.to_csv(sum_path)
        #summary_df_2.to_csv(sum_path_2)
        pbar.update(1)

    print(f"Done! Files saved to {output_dir}")
    return output_df, summary_df

#### (1) logFC
to decide the edges between sample and protein node

In [153]:
def do_biological_logfc(
    data: pd.DataFrame,
    design: pd.DataFrame,
    threshold: float = 0.1,
    alpha: float = 0.05,
    control: str|int = 0
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Identifies 'Radicals' based on the Volcano Plot regions:
    1  (Red):  logFC > threshold  AND  adj.P-value < alpha
    -1 (Blue): logFC < -threshold AND  adj.P-value < alpha
    0  (Gray): Does not meet both criteria.
    """
    # 1. Safety Check: Is the data log-scaled?
    max_val = np.percentile(data.values, 99)
    
    if max_val > 50:
        warnings.warn(f"Data appears to be raw counts (Max: {max_val:.2f}). Applying log2(x + 1) transformation.")
        # Apply log2 transformation: adding 1 avoids log(0) errors
        working_data = np.log2(data + 1)
        if not isinstance(working_data, pd.DataFrame):
            working_data = pd.DataFrame(working_data, index=data.index, columns=data.columns)
    else:
        working_data = data.copy()
    print(f"Working data shape: {working_data.shape}")
    
    # 2. Align data and design -> data_t[samples, genes]
    if all(s in working_data.columns for s in design.index[:5]):
        data_t = working_data.transpose()
    else:
        data_t = working_data
        working_data = working_data.transpose()
         
    print(f"Transpose data shape: {data_t.shape}, supposed to be [sample * gene]")

    control_idx = design[design['Target'] == control].index.to_list()
    case_idx = design[design['Target'] != control].index.to_list()
    
    # 3. Statistical Testing (Gene by Gene)
    results = []
    for gene in working_data.index:
        control_vals = working_data.loc[gene, control_idx]
        case_vals = working_data.loc[gene, case_idx]
        
        # Calculate Mean LogFC (Case Mean - Control Mean)
        mean_logfc = case_vals.mean() - control_vals.mean()
    
        # Handle zero variance to prevent NaNs
        if control_vals.var() == 0 and case_vals.var() == 0:
            p_val = 1.0
        else:
            # Perform Welch's T-Test (Comparing the two distributions): equal_var=False
            _, p_val = stats.ttest_ind(case_vals, control_vals, equal_var=False, nan_policy='omit')
            if np.isnan(p_val): p_val = 1.0
        
        results.append({'gene': gene, 'logFC': mean_logfc, 'p_value': p_val})
    
    stats_df = pd.DataFrame(results).set_index('gene')
    
    # 4. Multiple Testing Correction (FDR / Benjamini-Hochberg)
    # This prevents false positives when testing thousands of genes
    stats_df['adj_P_val'] = multipletests(stats_df['p_value'], method='fdr_bh')[1]

    # 5. Scoring the Samples (The 'Volcano' Logic)
    # Initialize output matrix [samples x genes]
    output_df = pd.DataFrame(0, index=data_t.index, columns=working_data.index)
    
    # We only mark a gene as 1 or -1 if the GENE ITSELF is significant overall
    # and the individual sample's expression is extreme.
    for gene in working_data.index:
        gene_stats = stats_df.loc[gene]
        
        if gene_stats['adj_P_val'] < alpha:
            # Calculate sample-specific deviation from control mean
            ctrl_mean = working_data.loc[gene, control_idx].mean()
            sample_deviations = data_t[gene] - ctrl_mean
            
            # Upper-Red Region (Significant Up)
            output_df.loc[sample_deviations > threshold, gene] = 1
            # Upper-Blue Region (Significant Down)
            output_df.loc[sample_deviations < -threshold, gene] = -1

    # 6. Create the Summary with counts (-1, 0, 1)
    # We transpose output_df back to [genes x counts] to match stats_df
    counts = output_df.apply(pd.Series.value_counts).fillna(0).astype(int).T
    
    # Ensure all three columns exist even if some aren't present in the data
    for col in [-1, 0, 1]:
        if col not in counts.columns:
            counts[col] = 0
            
    # Rename columns for clarity
    counts = counts.rename(columns={-1: 'count_neg', 0: 'count_neutral', 1: 'count_pos'})
    
    # Combine stats and counts
    summary_df = pd.concat([stats_df, counts[['count_neg', 'count_neutral', 'count_pos']]], axis=1)
    
    # 7. Metadata and Summary
    label_mapping = {key: val for val, key in enumerate(np.unique(design['Target']))}
    output_df['label'] = design.loc[output_df.index, 'Target'].map(label_mapping)
    
    
    return output_df, summary_df

Use Disease group as baseline, Identifies 'Radicals' based on the Volcano Plot regions:
+ 1  (Red):  logFC > threshold  AND  adj.P-value < alpha
+ -1 (Blue): logFC < -threshold AND  adj.P-value < alpha
+ 0  (Gray): Does not meet both criteria.

In [158]:
df_geo_logfc,summary_geo_logfc = process_and_save(
    data=df_geo, 
    design=geo_design, 
    threshold=0.1, 
    control=1,
    do_function=do_biological_logfc,
    output_dir='./data/GEO/GSE33000_ad_hd/map_health_kg',
    method = 'logfc'
)

Created directory: ./data/GEO/GSE33000_ad_hd/map_health_kg
Starting logfc analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Working data shape: (467, 17405)
Transpose data shape: (467, 17405), supposed to be [sample * gene]


Overall Progress: 100%|██████████| 2/2 [01:35<00:00, 47.81s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd/map_health_kg


Use Control group as baseline, Identifies 'Radicals' based on the Volcano Plot regions:
+ 1  (Red):  logFC > threshold  AND  adj.P-value < alpha
+ -1 (Blue): logFC < -threshold AND  adj.P-value < alpha
+ 0  (Gray): Does not meet both criteria.

In [159]:
summary_geo_logfc

,logFC,p_value,adj_P_val,count_neg,count_neutral,count_pos
A1BG,0.019171,8.956170e-04,1.284037e-03,15,414,38
A2M,-0.034544,1.000614e-07,1.832074e-07,51,387,29
A2ML1,-0.092729,2.125495e-32,1.287652e-31,107,316,44
A3GALT2,-0.005049,2.532798e-01,2.849234e-01,0,467,0
A4GALT,-0.336735,1.410309e-49,3.137060e-48,239,118,110
...,...,...,...,...,...,...
ZZEF1,0.004145,7.149134e-01,7.413757e-01,0,467,0
ZZZ3,-0.083465,6.411587e-25,2.589781e-24,109,324,34
tcag7.1017,0.109448,2.031649e-25,8.405241e-25,75,234,158
tcag7.216,0.008954,3.378552e-01,3.724818e-01,0,467,0


In [156]:
df_geo_logfc,summary_geo_logfc = process_and_save(
    data=df_geo, 
    design=geo_design, 
    threshold=0.1, 
    control=0,
    do_function=do_biological_logfc,
    output_dir='./data/GEO/GSE33000_ad_hd/map_ad_kg',
    method = 'logfc'
)
summary_geo_logfc

Starting logfc analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Working data shape: (467, 17405)
Transpose data shape: (467, 17405), supposed to be [sample * gene]


Overall Progress: 100%|██████████| 2/2 [01:31<00:00, 45.85s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd/ad_kg


,logFC,p_value,adj_P_val,count_neg,count_neutral,count_pos
A1BG,-0.019171,8.956170e-04,1.284037e-03,36,411,20
A2M,0.034544,1.000614e-07,1.832074e-07,17,375,75
A2ML1,0.092729,2.125495e-32,1.287652e-31,13,319,135
A3GALT2,0.005049,2.532798e-01,2.849234e-01,0,467,0
A4GALT,0.336735,1.410309e-49,3.137060e-48,64,91,312
...,...,...,...,...,...,...
ZZEF1,-0.004145,7.149134e-01,7.413757e-01,0,467,0
ZZZ3,0.083465,6.411587e-25,2.589781e-24,15,311,141
tcag7.1017,-0.109448,2.031649e-25,8.405241e-25,185,254,28
tcag7.216,-0.008954,3.378552e-01,3.724818e-01,0,467,0


In [90]:
df_geo_logfc

,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981,label
GSM1423780,-1,1,0,0,1,-1,0,-1,0,1,...,0,1,0,0,0,1,1,0,-1,1
GSM1423781,1,0,0,0,1,1,0,-1,0,-1,...,0,0,0,0,0,-1,1,0,-1,1
GSM1423782,0,0,-1,0,-1,1,-1,0,0,1,...,0,1,0,0,0,1,1,0,1,1
GSM1423783,0,1,0,0,1,1,1,-1,0,-1,...,0,-1,0,0,0,-1,-1,0,-1,1
GSM1423784,-1,0,0,0,-1,1,0,0,0,1,...,0,1,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1424242,-1,-1,-1,0,0,0,0,1,0,1,...,0,0,0,0,0,1,0,0,1,0
GSM1424243,1,0,-1,0,0,0,1,1,0,0,...,0,-1,0,0,0,-1,1,0,-1,0
GSM1424244,1,-1,-1,0,1,1,0,0,0,1,...,0,-1,0,0,0,1,1,0,0,0
GSM1424245,1,1,-1,0,1,0,0,0,0,0,...,0,-1,0,0,0,0,1,0,0,0


#### (2) ECDF

In [157]:
from typing import Any

import pandas._typing as pdt
import numpy.typing as npt
from scipy.interpolate import interp1d
import pandera.typing as pat
from statsmodels.distributions.empirical_distribution import ECDF

def do_radical_search(
        data: pd.DataFrame,
        design: pd.DataFrame,
        threshold: float,
        control: str|int,
        control_based: bool
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Identify the samples with extreme feature values either based on the entire dataset or control population.

    :param data: Dataframe containing the gene expression values
    :param design: Dataframe containing the design table for the data
    :param threshold: Threshold for choosing patients that are "extreme" w.r.t. the controls
    :param control: label used for representing the control in the design table of the data
    :param control_based: The scoring is based on the control population instead of entire dataset
    :return: Dataframe containing the Single Sample scores using radical searching
    """
    # Transpose matrix to get the patients as the rows
    if all(s in data.columns for s in design.index[:5]):
        data_transpose = data.transpose()
    else:
        data_transpose = data

    # Give each label an integer to represent the labels during classification
    label_mapping = {
        key: val
        for val, key in enumerate(np.unique(design['Target']))
    }

    # Make sure the number of rows of transposed data and design are equal
    assert len(data_transpose) == len(design), 'Data doesnt match the design matrix'

    # Create a dataframe initialized with 0's [patients x features]
    output_df = pd.DataFrame(0, index=data_transpose.index, columns=data_transpose.columns)

    # Values that are greater than the threshold or lesser than negative threshold are considered as extremes.
    upper_thresh = 1 - (threshold / 100)
    lower_thresh = (threshold / 100)

    if control_based:
        controls = data_transpose[list(design.Target == control)]

        # Calculate the empirical cdf for every gene and get the cdf score for the data
        control_ecdf = controls.apply(_get_ecdf, step=False, extrapolate=True).values
        cdf_score = _apply_func(data_transpose, control_ecdf).fillna(0)

        # Check if each patient's feature is over or under expressed compared to the control population
        output_df = pd.DataFrame(np.where(cdf_score.values > upper_thresh, 1, output_df.values))
        output_df = pd.DataFrame(np.where(cdf_score.values < lower_thresh, -1, output_df.values))

    else:
        # Calculate the empirical cdf for every gene and get the cdf score for the data
        feature_to_ecdf = {
            feature: _get_ecdf(data_transpose[feature].values)
            for feature in data_transpose
            if len(data_transpose[feature].unique()) > 1  # Check not all values are the same
        }

        # Iterate over patients and check if any of its features is significant
        for patient_index, features in data_transpose.iterrows():

            # Iterate over patient features
            for feature, value in features.items():

                # Skip if feature has no calculated eCDF
                if feature not in feature_to_ecdf:
                    continue

                # Calculate position of the patient in the distribution of the feature
                patient_position_in_distribution = float(feature_to_ecdf[feature]([value])[0])

                if patient_position_in_distribution <= lower_thresh:
                    output_df.loc[patient_index, feature] = -1

                if patient_position_in_distribution > upper_thresh:
                    output_df.loc[patient_index, feature] = 1

    output_df.columns = data.index
    output_df.index = data.columns

    summary_df = output_df.apply(pd.Series.value_counts)

    # Add labels to the data samples
    label = design['Target'].map(label_mapping)
    label.reset_index(drop=True, inplace=True)

    output_df['label'] = label.values

    return output_df, summary_df


def _get_ecdf(
        obs: pdt.ArrayLike,
        side: str = 'right',
        step: bool = True,
        extrapolate: bool = False
) -> Any:
    """Calculate the Empirical CDF of an array and return it as a function.

    :param obs: Observations
    :param side: Defines the shape of the intervals constituting the steps. 'right' correspond to [a, b) intervals
        and 'left' to (a, b]
    :param step: Boolean value to indicate if the returned value must be a step function or an continuous based on
        interpolation or extrapolation function
    :param extrapolate: Boolean value to indicate if the continuous must be based on extrapolation
    :return: Empirical CDF as a function
    """
    if step:
        return ECDF(x=obs, side=side)
    else:
        obs = np.array(obs, copy=True)
        obs.sort()

        num_of_obs = len(obs)

        y = np.linspace(1. / num_of_obs, 1, num_of_obs)

        if extrapolate:
            return interp1d(obs, y, bounds_error=False, fill_value="extrapolate")  # type: ignore
        else:
            return interp1d(obs, y)
def _apply_func(
        df: pd.DataFrame,
        func_list: npt.NDArray[Any]
) -> pd.DataFrame:
    """Apply functions from the list (in order) on the respective column.

    :param df: Data on which the functions need to be applied
    :param func_list: List of functions to be applied
    :return: Dataframe which has been processed
    """
    final_df = pd.DataFrame()

    new_columns = [index for index, _ in enumerate(df.columns)]
    old_columns = list(df.columns)

    df.columns = pd.Index(new_columns)

    for idx, i in enumerate(tqdm(df.columns, desc='Searching for radicals: ')):
        final_df[i] = np.apply_along_axis(func_list[idx], 0, df[i].values)  # type: ignore

    final_df.columns = pd.Index(old_columns)

    return final_df


Disease group as baseline

In [ ]:
df_adni_ecdf,summary_adni_ecdf = process_and_save(
    data=df_adni_2cls.transpose(), 
    design=adni_design_2cls, 
    threshold=5, 
    control=1,
    do_function=do_radical_search,
    output_dir='./data/ADNI/old_target',
    method = 'ecdf',
    control_based=True
)
summary_adni_ecdf

Created directory: ./data/ADNI/map_health_kg
Starting ecdf analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_85381/4089639019.py:147: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[i] = np.apply_along_axis(func_list[idx], 0, df[i].values)  # type: ignore
Overall Progress: 100%|██████████| 2/2 [01:34<00:00, 47.07s/it]

Done! Files saved to ./data/ADNI/map_health_kg


genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
-1,22,21,22,25,20,20,26,19,27,28,...,16,14,19,30,25,19,17,28,18,16
0,411,408,414,403,413,402,407,415,407,406,...,409,416,401,406,414,416,418,408,416,414
1,22,26,19,27,22,33,22,21,21,21,...,30,25,35,19,16,20,20,19,21,25


In [162]:
df_geo_ecdf,summary_geo_ecdf = process_and_save(
    data=df_geo.transpose(), 
    design=geo_design, 
    threshold=5, 
    control=1,
    do_function=sample_scoring.do_radical_search,
    output_dir='./data/GEO/GSE33000_ad_hd/map_health_kg',
    method = 'ecdf',
    control_based=True
)
summary_geo_ecdf

Starting ecdf analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]/Users/yuxiaoxuan/master_thesis/FireGNN/utils/sample_scoring.py:154: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  old_columns = list(df.columns)
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
Overall Progress: 100%|██████████| 2/2 [01:11<00:00, 35.65s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd/map_health_kg


,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDB,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981
-1,16,27,48,19,96,28,37,15,18,15,...,19,18,15,20,18,17,42,15,19,15
0,427,424,403,430,355,418,413,391,432,414,...,405,432,369,422,429,432,409,417,430,354
1,24,16,16,18,16,21,17,61,17,38,...,43,17,83,25,20,18,16,35,18,98


Control group as baseline

In [94]:
df_adni_ecdf,summary_adni_ecdf = process_and_save(
    data=df_adni_2cls.transpose(), 
    design=adni_design_2cls, 
    threshold=5, 
    control=0,
    do_function=do_radical_search,
    output_dir='./data/ADNI',
    method = 'ecdf',
    control_based=True
)
summary_adni_ecdf

Starting ecdf analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]/Users/yuxiaoxuan/master_thesis/FireGNN/utils/sample_scoring.py:154: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
Overall Progress: 100%|██████████| 2/2 [01:21<00:00, 40.80s/it]

Done! Files saved to ./data/ADNI


genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
-1,21,26,21,19,30,25,22,27,15,16,...,34,25,26,20,19,23,29,19,24,32
0,408,408,404,413,399,414,408,402,414,410,...,404,407,414,408,407,406,392,407,393,400
1,26,21,30,23,26,16,25,26,26,29,...,17,23,15,27,29,26,34,29,38,23


In [98]:
df_geo_ecdf,summary_geo_ecdf = process_and_save(
    data=df_geo.transpose(), 
    design=geo_design, 
    threshold=5, 
    control=0,
    do_function=sample_scoring.do_radical_search,
    output_dir='./data/GEO/GSE33000_ad_hd',
    method = 'ecdf',
    control_based=True
)
summary_geo_ecdf

Starting ecdf analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]/Users/yuxiaoxuan/master_thesis/FireGNN/utils/sample_scoring.py:154: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  old_columns = list(df.columns)
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
Overall Progress: 100%|██████████| 2/2 [01:02<00:00, 31.40s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd


,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDB,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981
-1,85,16,9,41,7,13,17,142,27,131,...,40,32,105,33,29,43,12,106,45,133
0,357,366,293,380,325,412,376,316,396,321,...,414,360,353,413,399,394,364,343,381,325
1,25,85,165,46,135,42,74,9,44,15,...,13,75,9,21,39,30,91,18,41,9


#### (3) STD / Z-score

In [134]:
def do_std(
    data: pd.DataFrame,
    design: pd.DataFrame,
    threshold: float,
    control: str|int
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Identifies 'Radicals' based on the Standard Deviation of the Control group.
    1  : Sample_Value > (Control_Mean + threshold * Control_Std)
    -1 : Sample_Value < (Control_Mean - threshold * Control_Std)
    0  : Within the expected variation of Controls
    
    :param data: Dataframe gene expression values (assumed log-scaled)
    :param design: Dataframe [samples x info] containing the 'Target' column
    :param threshold: std multiplier
    :param control: The string or int label for the control group
    """

    # 1. Safety Check: Is the data log-scaled?
    max_val = np.percentile(data.values, 99)
    
    if max_val > 50:
        warnings.warn(f"Data appears to be raw counts (Max: {max_val:.2f}). Applying log2(x + 1) transformation.")
        # Apply log2 transformation: adding 1 avoids log(0) errors
        working_data = np.log2(data + 1)
        if not isinstance(working_data, pd.DataFrame):
            working_data = pd.DataFrame(working_data, index=data.index, columns=data.columns)
    else:
        working_data = data.copy()
    print(f"Working data shape: {working_data.shape}")
    
    # 2. Align data and design -> data_t[samples, genes]
    if all(s in working_data.columns for s in design.index[:5]):
        data_t = working_data.transpose()
    else:
        data_t = working_data 
    print(f"Transpose data shape: {data_t.shape}, supposed to be [sample * gene]")

    # 3. Calculate the Mean and std of the Control Group for every gene
    control_samples = design[design['Target'] == control].index
    valid_controls = control_samples.intersection(data_t.index)
    
    # Calculate stats across the control rows
    control_stats = data_t.loc[valid_controls].agg(['mean', 'std'], axis=0).T
    control_stats['std'] = control_stats['std'].replace(0, 1e-6)
    
    # 4. Scoring Logic (Vectorized)
    # Reindex bounds to match data_t columns exactly
    upper_bound = control_stats['mean'] + (threshold * control_stats['std'])
    lower_bound = control_stats['mean'] - (threshold * control_stats['std'])

    # Efficiently create the -1, 0, 1 matrix
    output_df = pd.DataFrame(0, index=data_t.index, columns=data_t.columns)
    output_df[data_t > upper_bound] = 1
    output_df[data_t < lower_bound] = -1

    # 5. Summary Statistics (Safe Alignment)
    summary_df = control_stats.copy().rename(columns={'mean': 'control_mean', 'std': 'control_std'})
    
    # Count occurrences of -1, 0, 1 for each gene
    # value_counts() on the columns of output_df
    counts = output_df.apply(lambda x: x.value_counts()).fillna(0).T
    
    # Map the count columns specifically using reindex to avoid NaNs
    summary_df['count_neg'] = counts.reindex(columns=[-1]).iloc[:, 0].fillna(0).astype(int)
    summary_df['count_neutral'] = counts.reindex(columns=[0]).iloc[:, 0].fillna(0).astype(int)
    summary_df['count_pos'] = counts.reindex(columns=[1]).iloc[:, 0].fillna(0).astype(int)

    # 6. Add labels
    label_mapping = {key: val for val, key in enumerate(np.unique(design['Target']))}
    output_df['label'] = design.loc[output_df.index, 'Target'].map(label_mapping)
    
    summary_df_1 = output_df.apply(pd.Series.value_counts)
    
    return output_df, summary_df_1

Disease group as baseline

In [163]:
df_adni_std,summary_adni_std = process_and_save(
    data=df_adni_2cls, 
    design=adni_design_2cls, 
    threshold=1.5, 
    control=1,
    do_function=do_std,
    output_dir="./data/ADNI/map_health_kg",
    method='std'
)

Starting std analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Working data shape: (455, 19100)
Transpose data shape: (455, 19100), supposed to be [sample * gene]


Overall Progress: 100%|██████████| 2/2 [00:31<00:00, 15.58s/it]

Done! Files saved to ./data/ADNI/map_health_kg


In [164]:
df_geo_std,summary_geo_std = process_and_save(
    data=df_geo, 
    design=geo_design, 
    threshold=2, 
    control=1,
    do_function=do_std,
    output_dir="./data/GEO/GSE33000_ad_hd/map_health_kg",
    method='std'
)

Starting std analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Working data shape: (467, 17405)
Transpose data shape: (467, 17405), supposed to be [sample * gene]


Overall Progress: 100%|██████████| 2/2 [00:29<00:00, 14.98s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd/map_health_kg


Control group as baseline

In [137]:
df_adni_std,summary_adni_std = process_and_save(
    data=df_adni_2cls, 
    design=adni_design_2cls, 
    threshold=1.5, 
    control=0,
    do_function=do_std,
    output_dir="./data/ADNI",
    method='std'
)

Starting std analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Working data shape: (455, 19100)
Transpose data shape: (455, 19100), supposed to be [sample * gene]


Overall Progress: 100%|██████████| 2/2 [00:33<00:00, 16.66s/it]

Done! Files saved to ./data/ADNI


In [138]:
summary_adni_std

genes,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,label
-1,25,25,26,27,32,25,22,34,29,23,...,25,26,27,21,21,36,34,36,43,NaN
0,395,399,397,395,387,406,399,398,396,406,...,393,411,400,396,395,379,399,383,387,222.0
1,35,31,32,33,36,24,34,23,30,26,...,37,18,28,38,39,40,22,36,25,233.0


In [139]:
df_geo_std,summary_geo_std = process_and_save(
    data=df_geo, 
    design=geo_design, 
    threshold=2, 
    control=0,
    do_function=do_std,
    output_dir="./data/GEO/GSE33000_ad_hd",
    method='std'
)

Starting std analysis...


Overall Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Working data shape: (467, 17405)
Transpose data shape: (467, 17405), supposed to be [sample * gene]


Overall Progress: 100%|██████████| 2/2 [00:28<00:00, 14.34s/it]

Done! Files saved to ./data/GEO/GSE33000_ad_hd


In [140]:
summary_geo_std

,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981,label
-1,34,11,7,15,1,9,1.0,110,14,115,...,26,100.0,12,10,12,6,99,31,143,NaN
0,413,403,346,427,325,447,466.0,354,424,350,...,409,367.0,443,423,444,374,361,423,323,157.0
1,20,53,114,25,141,11,NaN,3,29,2,...,32,NaN,12,34,11,87,7,13,1,310.0


### 2. Sample_Network Generation

In [4]:
from collections import defaultdict
from data_processing.pyg_graph_generator import add_patient_attrs, build_knn_graph_with_masks
import re

In [17]:
import pandas._typing as pdt
import argparse
import os
from typing import Callable, Optional, List, Tuple, Any
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
import numpy as np
import numpy.typing as npt
import pandas as pd
import pandas._typing as pdt
from scipy.interpolate import interp1d
from statsmodels.distributions.empirical_distribution import ECDF
from tqdm import tqdm
import pandera.typing as pat

def do_radical_search(
        data: pd.DataFrame,
        design: pd.DataFrame,
        threshold: float,
        control: str|int,
        control_based: bool
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Identify the samples with extreme feature values either based on the entire dataset or control population.

    :param data: Dataframe containing the gene expression values
    :param design: Dataframe containing the design table for the data
    :param threshold: Threshold for choosing patients that are "extreme" w.r.t. the controls
    :param control: label used for representing the control in the design table of the data
    :param control_based: The scoring is based on the control population instead of entire dataset
    :return: Dataframe containing the Single Sample scores using radical searching
    """
    # Transpose matrix to get the patients as the rows
    if all(s in data.columns for s in design.index[:5]):
        data_transpose = data.transpose()
    else:
        data_transpose = data.copy()
    print(f"Transposed data shape: {data_transpose.shape}, supposed to be [sample, gene]")
    # Give each label an integer to represent the labels during classification
    label_mapping = {
        key: val
        for val, key in enumerate(np.unique(design['Target']))
    }

    # Make sure the number of rows of transposed data and design are equal
    assert len(data_transpose) == len(design), 'Data doesnt match the design matrix'

    # Create a dataframe initialized with 0's [patients x features]
    output_df = pd.DataFrame(0, index=data_transpose.index, columns=data_transpose.columns)
    # print('output_df columns:', output_df.columns)
    # Values that are greater than the threshold or lesser than negative threshold are considered as extremes.
    upper_thresh = 1 - (threshold / 100)
    lower_thresh = (threshold / 100)

    if control_based:
        controls = data_transpose[list(design.Target == control)]

        # Calculate the empirical cdf for every gene and get the cdf score for the data
        control_ecdf = controls.apply(_get_ecdf, step=False, extrapolate=True).values
        cdf_score = _apply_func(data_transpose, control_ecdf).fillna(0)

        # Check if each patient's feature is over or under expressed compared to the control population
        # output_df = pd.DataFrame(np.where(cdf_score.values > upper_thresh, 1, output_df.values))
        # output_df = pd.DataFrame(np.where(cdf_score.values < lower_thresh, -1, output_df.values))
        conditions = [
            cdf_score.values > upper_thresh,
            cdf_score.values < lower_thresh
        ]
        
        # 2. Define the choices corresponding to those conditions
        choices = [1, -1]
        
        # 3. Use np.select to build the grid, defaulting remaining values to 0
        new_values = np.select(conditions, choices, default=0)
        
        # 4. Safely wrap it back into a DataFrame while strictly preserving your metadata
        output_df = pd.DataFrame(new_values, index=data_transpose.index, columns=data_transpose.columns)

    else:
        # Calculate the empirical cdf for every gene and get the cdf score for the data
        feature_to_ecdf = {
            feature: _get_ecdf(data_transpose[feature].values)
            for feature in data_transpose
            if len(data_transpose[feature].unique()) > 1  # Check not all values are the same
        }

        # Iterate over patients and check if any of its features is significant
        for patient_index, features in data_transpose.iterrows():

            # Iterate over patient features
            for feature, value in features.items():

                # Skip if feature has no calculated eCDF
                if feature not in feature_to_ecdf:
                    continue

                # Calculate position of the patient in the distribution of the feature
                patient_position_in_distribution = float(feature_to_ecdf[feature]([value])[0])

                if patient_position_in_distribution <= lower_thresh:
                    output_df.loc[patient_index, feature] = -1

                if patient_position_in_distribution > upper_thresh:
                    output_df.loc[patient_index, feature] = 1

    if all(s in data.columns for s in design.index[:5]):
        # Data was transposed: Rows are samples (data.columns), Columns are genes (data.index)
        output_df.index = data.columns
        output_df.columns = data.index
    else:
        # Data was NOT transposed
        output_df.index = data.index
        output_df.columns = data.columns

    print('output_df columns:', output_df.columns)
    summary_df = output_df.apply(pd.Series.value_counts)

    # Add labels to the data samples
    label = design['Target'].map(label_mapping)

    output_df['label'] = output_df.index.map(label)

    return output_df, summary_df


def _get_ecdf(
        obs: pdt.ArrayLike,
        side: str = 'right',
        step: bool = True,
        extrapolate: bool = False
) -> Any:
    """Calculate the Empirical CDF of an array and return it as a function.

    :param obs: Observations
    :param side: Defines the shape of the intervals constituting the steps. 'right' correspond to [a, b) intervals
        and 'left' to (a, b]
    :param step: Boolean value to indicate if the returned value must be a step function or an continuous based on
        interpolation or extrapolation function
    :param extrapolate: Boolean value to indicate if the continuous must be based on extrapolation
    :return: Empirical CDF as a function
    """
    if step:
        return ECDF(x=obs, side=side)
    else:
        obs = np.array(obs, copy=True)
        obs.sort()

        num_of_obs = len(obs)

        y = np.linspace(1. / num_of_obs, 1, num_of_obs)

        if extrapolate:
            return interp1d(obs, y, bounds_error=False, fill_value="extrapolate")  # type: ignore
        else:
            return interp1d(obs, y)


def _apply_func(
        df: pd.DataFrame,
        func_list: npt.NDArray[Any]
) -> pd.DataFrame:
    """Apply functions from the list (in order) on the respective column."""
    final_df = pd.DataFrame(index=df.index) # Keep the sample index intact

    # Use enumerate on original columns without renaming them
    for idx, col_name in enumerate(tqdm(df.columns, desc='Searching for radicals: ')):
        final_df[col_name] = np.apply_along_axis(func_list[idx], 0, df[col_name].values)  # type: ignore

    return final_df

In [14]:
exp_path = "../data/ADNI/adni_exp_realcleaned.csv"
design_path = "../data/ADNI/design_with_real_target.tsv"

# 1. Load expression df, smaple scoring df, KG
exp_df = pd.read_csv(exp_path, index_col=0)
design = pd.read_csv(design_path, index_col=0, sep='\t')
design['Target'] = design['Old_Target'].map({"Control":0, "Disease":1})
design

,Old_Target,Target
FileName,,
116_S_1249,Control,0
037_S_4410,Control,0
006_S_4153,Disease,1
116_S_1232,Control,0
099_S_4205,Disease,1
...,...,...
009_S_2381,Disease,1
053_S_4557,Disease,1
073_S_4300,Disease,1


In [ ]:
sample_scoring, summary = do_radical_search(exp_df, design,1, control=0, control_based=True)

Transposed data shape: (744, 50788), supposed to be [sample, gene]


Searching for radicals:   0%|          | 0/50788 [00:00<?, ?it/s]/tmp/ipykernel_634984/3295207.py:167: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col_name] = np.apply_along_axis(func_list[idx], 0, df[col_name].values)  # type: ignore
/tmp/ipykernel_634984/3295207.py:167: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[col_name] = np.apply_along_axis(func_list[idx], 0, df[col_name].values)  # type: ignore
/tmp/ipykernel_634984/3295207.py:167: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

In [7]:
class PatientNetworkGenerator:
    def __init__(self, kg_disease, kg_healthy):
        """
        Initializes with knowledge graphs. All input graphs are forced to MultiDiGraph.
        """
        # Ensure all base KGs are MultiDiGraph to support multiple relations
        self.kg_disease = nx.MultiDiGraph(kg_disease)
        self.kg_healthy = nx.MultiDiGraph(kg_healthy)
        
        self.relation_map = {1: 'up_reg', -1: 'down_reg'}
        self.pattern_hgnc =  r'^p\(HGNC:"([^"]+)"\)$'
        self.pattern_uniprotkb =  r'^p\(UniProtKB:"([^"_%]+)_[A-Z]+"\)$'
        self.pattern_dash = r'(\w+)_HUMAN'

    def gene_symbol_extractor(self, text, pattern:str):
        # ^ ensures start at the beginning, $ ensures end at the ')'
        match = re.search(pattern, text)
        if match:
            target = match.group(1)
            return target.upper()
        return None

    def get_symbol_mapping(self, graph):
        """
        Universal helper function to create {gene_symbol: [kg_nodes]} mapping 
        across ad_kg, control_kg, ppi_kg, and prime_kg.
        """
        mapping = defaultdict(list)
        if graph is None: 
            return dict(mapping)
        
        for node, attr in graph.nodes(data=True):
            # Ensure node is a string and skip empty/complex modifications
            if not isinstance(node, str) or any(x in node for x in ['frag(', 'var(', 'pmod(', 'loc(']):
                continue
                
            symbol = None
            
            # Case 1: BEL Namespace syntax (ad_kg & control_kg) -> p(HGNC:"...") or p(UniProtKB:"...")
            if node.startswith('p(') and node.endswith(')'):
                if 'HGNC' in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_hgnc)
                elif 'UniProtKB' in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_uniprotkb)
                else:
                    # Fallback for unexpected namespaces inside p(...) like p(FIXME:"...")
                    match = re.search(r'["\']([^"\']+)["\']', node)
                    if match:
                        symbol = match.group(1)
                    #continue

            # Case 2: handle (ppi_kg) -> AL1A1_HUMAN
            elif '_' in node and any(suffix in node for suffix in ['_HUMAN']):
                if 'UniProtKB' not in node and 'HGNC' not in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_dash)

            else:
                # Case 3: handle Gene/Protein symbols in PrimeKg
                clean_node = node.strip().upper()
                if attr.get('label') and attr.get('label') in 'Gene/Protein':
                    symbol = clean_node

            # Append to our list mapping if a valid symbol was resolved
            if symbol:
                symbol_upper = symbol.upper()
                mapping[symbol_upper].append(node)
                
        return dict(mapping)
    
    def generate(self, 
                 data: pd.DataFrame, 
                 exp_df: pd.DataFrame, 
                 base_graph:str='disease') -> Tuple[nx.MultiGraph, pd.DataFrame]:
        """
        Connects samples to KG protein nodes using base_graph.
        """
        patient_labels = data['label'].to_dict()
        scores = data.drop(columns=['label'])
        
        # 1. Force base_graph to MultiDiGraph and copy
        if base_graph == 'disease':
            self.base_graph = self.kg_disease
        else:
            self.base_graph = self.kg_healthy
        
        self.relation_map = {1: f'up_reg', -1: f'down_reg'}

        overlay_graph = nx.MultiDiGraph(self.base_graph).copy()
        
        symbol_to_kg_node = self.get_symbol_mapping(overlay_graph)
        common_proteins = [s for s in scores.columns if s in symbol_to_kg_node]
        
        sparse_data = scores[common_proteins].stack()
        radicals = sparse_data[sparse_data != 0]

        summary_df = pd.DataFrame(0, index=data.index, columns=['pos_edges', 'neg_edges'])

        summary_df['linked_nodes'] = [[] for _ in range(len(summary_df))] # Initialize with empty lists
        summary_df['label'] = data['label'].to_list()

        # add patient nodes to overlay graph
        overlay_graph = add_patient_attrs(G=overlay_graph,
                                          features_df=exp_df,
                                          labels_series=data['label'],
                                          )
        # add edges between patient and protein
        for key, val in tqdm(radicals.items(), total=len(radicals), desc="Linking Samples"):
            if not (isinstance(key, tuple) and len(key) == 2):
            # unexpected index shape — skip or handle explicitly
                continue
            patient, symbol = key

            for protein_node in symbol_to_kg_node[symbol]:
                rel = self.relation_map.get(int(val))
                
                if not overlay_graph.has_node(patient):
                    overlay_graph.add_node(patient, label=patient_labels[patient], type='Patient')

                weight_value = float(exp_df.loc[patient, symbol])
                
                # Use relation as the 'key' for MultiDiGraph edges
                overlay_graph.add_edge(patient, protein_node, relation=rel, weight=weight_value, label=patient_labels[patient])
                overlay_graph.add_edge(protein_node, patient, relation=f'rev_{rel}', weight=weight_value,label=patient_labels[patient])
                
                col = 'pos_edges' if int(val) == 1 else 'neg_edges'
                summary_df.at[patient, col] += 1
                summary_df.at[patient,'linked_nodes'].append(protein_node)

        return overlay_graph, summary_df

In [8]:
# get kg_df
kg_disease = load_graph("../datasets/base_kgs/oldcleaned_ppi_kg.pkl")
kg_control = load_graph("../data/KG/healthy_aging_reversed_remove_noncausal.pkl")
exp_path = "../data/ADNI/adni_exp_realcleaned.csv"
scoring_path = "../CLEP_repeat/results/realADNI/PPI_KGs/ecdf_1/sample_scoring_ecdf.csv"

# 1. Load expression df, smaple scoring df, KG
exp_df = pd.read_csv(exp_path, index_col=0)
if exp_df.shape[0] > exp_df.shape[1]:
    exp_df = exp_df.transpose()
data = pd.read_csv(scoring_path, index_col=0)

# clean exp_df before K-NN
# drop genes with no variation
exp_df = exp_df.loc[:, exp_df.std() > 0]
# Using median is usually safer for gene expression
exp_df = exp_df.fillna(exp_df.median())
# normalize safely
min_val = exp_df.min()
max_val = exp_df.max()
exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
# final fill-na
exp_norm = exp_norm.fillna(0)

# 2. Generate PatientNetwork
png = PatientNetworkGenerator(kg_disease=kg_disease,
                                kg_healthy=kg_control)

network, summary = png.generate(data=data,
                                exp_df=exp_norm,
                                base_graph='disease')

Loaded graph from ../datasets/base_kgs/oldcleaned_ppi_kg.pkl: 8910 nodes, 209730 edges
Loaded graph from ../data/KG/healthy_aging_reversed_remove_noncausal.pkl: 4161 nodes, 13775 edges


Linking Samples: 0it [00:00, ?it/s]


In [10]:
data

,0,1,2,3,4,5,6,7,8,9,...,50779,50780,50781,50782,50783,50784,50785,50786,50787,label
116_S_1249,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
037_S_4410,0,0,0,-1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
006_S_4153,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
116_S_1232,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
099_S_4205,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
009_S_2381,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,1
053_S_4557,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
073_S_4300,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
041_S_4014,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


##### (1) generate adkg graphs
+ all patients link to proteins in ADKG
+ adni_scoring_method_graphs

In [ ]:
adni_ecdf_graph, summary = generat_and_save_hybrid(exp_path="./data/ADNI/adni_exp_2cls.csv",
                                            scoring_path="./data/ADNI/map_ad_kg/sample_scoring_ecdf.csv",
                                            kg_path='./data/KG/ad_network_with_reverse_edges.pkl',
                                            output_dir="./data/ADNI/map_ad_kg",
                                            kg_type='AD',
                                            scoring_method='ecdf',
                                            )

Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Matched 735 proteins between data and KG.


Linking Samples: 100%|██████████| 34243/34243 [00:03<00:00, 9657.14it/s] 


Saved graph to ./data/ADNI/map_ad_kg/G_patient_ADKG_ecdf.pkl: 4187 nodes, 83279 edges


In [29]:
adni_std_graph, summary = generat_and_save(exp_path="./data/ADNI/adni_exp_2cls.csv",
                                            scoring_path="./data/ADNI/map_ad_kg/sample_scoring_std.csv",
                                            kg_path='./data/KG/ad_network_with_reverse_edges.pkl',
                                            output_dir="./data/ADNI/map_ad_kg",
                                            kg_type='AD',
                                            scoring_method='std',
                                            pattern=pattern_hgnc)

Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Matched 735 proteins between data and KG.


Linking Samples: 100%|██████████| 40918/40918 [00:03<00:00, 10799.41it/s]


Saved graph to ./data/ADNI/map_ad_kg/G_patient_ADKG_std.pkl: 4187 nodes, 96629 edges


+ generate geo_adkg graphs
+ all patients link to proteins in ADKG

In [51]:
geo_ecdf_graph, summary = generat_and_save(exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_ad_kg/sample_scoring_ecdf.csv",
                                            kg_path='./data/KG/ad_network_with_reverse_edges.pkl',
                                            output_dir="./data/GEO/GSE33000_ad_hd/map_ad_kg",
                                            kg_type='AD',
                                            scoring_method='ecdf',
                                            pattern=pattern_hgnc)
geo_std_graph, summary = generat_and_save(exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_ad_kg/sample_scoring_std.csv",
                                            kg_path='./data/KG/ad_network_with_reverse_edges.pkl',
                                            output_dir="./data/GEO/GSE33000_ad_hd/map_ad_kg",
                                            kg_type='AD',
                                            scoring_method='std',
                                            pattern=pattern_hgnc)
geo_logfc_graph, summary = generat_and_save(exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_ad_kg/sample_scoring_logfc.csv",
                                            kg_path='./data/KG/ad_network_with_reverse_edges.pkl',
                                            output_dir="./data/GEO/GSE33000_ad_hd/map_ad_kg",
                                            kg_type='AD',
                                            scoring_method='logfc',
                                            pattern=pattern_hgnc)

Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Matched 694 proteins between data and KG.


Linking Samples: 100%|██████████| 71609/71609 [00:10<00:00, 6670.32it/s]


Saved graph to ./data/GEO/GSE33000_ad_hd/map_ad_kg/G_patient_ADKG_ecdf.pkl: 4199 nodes, 158011 edges
Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Matched 694 proteins between data and KG.


Linking Samples: 100%|██████████| 51617/51617 [00:07<00:00, 7077.85it/s]


Saved graph to ./data/GEO/GSE33000_ad_hd/map_ad_kg/G_patient_ADKG_std.pkl: 4195 nodes, 118027 edges
Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Matched 694 proteins between data and KG.


Linking Samples: 100%|██████████| 117598/117598 [00:17<00:00, 6810.78it/s]


Saved graph to ./data/GEO/GSE33000_ad_hd/map_ad_kg/G_patient_ADKG_logfc.pkl: 4199 nodes, 249989 edges


##### (2) generate patient_HealthKG graphs
+ all patients link to proteins in Healthy_Aging_KG
+ scoring functions (ecdf, logfc, std) use Disease group patients as baselines to search 'radical' points compared to Disease-Expression-Profile,
+ adni_graphs

In [40]:
# Pattern explanation:
# ^p\(UniProtKB:"  -> Starts with p(UniProtKB:"
# ([^"_%]+)        -> Capture group 1: The gene symbol (stops at _ or ")
# _[A-Z]+          -> Matches the species suffix (e.g., _HUMAN) but doesn't capture it
# "\)$             -> Ends strictly with ")
pattern_uniprotkb = r'^p\(UniProtKB:"([^"_%]+)_[A-Z]+"\)$'

In [42]:
adni_health_ecdf_graph, summary = generat_and_save(exp_path="./data/ADNI/adni_exp_2cls.csv",
                                            scoring_path="./data/ADNI/map_health_kg/sample_scoring_ecdf.csv",
                                            kg_path='./data/KG/healthy_aging_kg.pkl',
                                            output_dir="./data/ADNI/map_ad_kg",
                                            kg_type='Healthy',
                                            scoring_method='ecdf', 
                                            pattern=pattern_uniprotkb)
adni_health_std_graph, summary = generat_and_save(exp_path="./data/ADNI/adni_exp_2cls.csv",
                                            scoring_path="./data/ADNI/map_health_kg/sample_scoring_std.csv",
                                            kg_path='./data/KG/healthy_aging_kg.pkl',
                                            output_dir="./data/ADNI/map_ad_kg",
                                            kg_type='Healthy',
                                            scoring_method='std',
                                            pattern=pattern_uniprotkb)

Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Matched 210 proteins between data and KG.


Linking Samples: 100%|██████████| 9791/9791 [00:00<00:00, 9816.17it/s] 


Saved graph to ./data/ADNI/map_ad_kg/G_patient_HealthyKG_ecdf.pkl: 4616 nodes, 26406 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Matched 210 proteins between data and KG.


Linking Samples: 100%|██████████| 11981/11981 [00:01<00:00, 10495.56it/s]


Saved graph to ./data/ADNI/map_ad_kg/G_patient_HealthyKG_std.pkl: 4616 nodes, 30786 edges


+ all patients link to proteins in Healthy_Aging_KG
+ scoring functions (ecdf, logfc, std) use Disease group patients as baselines to search 'radical' points compared to Disease-Expression-Profile,
+ geo_graphs

In [52]:
geo_health_ecdf_graph, summary = generat_and_save(exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_health_kg/sample_scoring_ecdf.csv",
                                            kg_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern=pattern_uniprotkb,
                                            output_dir="./data/GEO/GSE33000_ad_hd/map_ad_kg",
                                            kg_type='Healthy',
                                            scoring_method='ecdf')
geo_health_std_graph, summary = generat_and_save(exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_health_kg/sample_scoring_std.csv",
                                            kg_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern=pattern_uniprotkb,
                                            output_dir="./data/GEO/GSE33000_ad_hd/map_ad_kg",
                                            kg_type='Healthy',
                                            scoring_method='std')
geo_health_logfc_graph, summary = generat_and_save(exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_health_kg/sample_scoring_logfc.csv",
                                            kg_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern=pattern_uniprotkb,
                                            output_dir="./data/GEO/GSE33000_ad_hd/map_ad_kg",
                                            kg_type='Healthy',
                                            scoring_method='logfc')

Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Matched 201 proteins between data and KG.


Linking Samples: 100%|██████████| 13039/13039 [00:01<00:00, 7333.96it/s]


Saved graph to ./data/GEO/GSE33000_ad_hd/map_ad_kg/G_patient_HealthyKG_ecdf.pkl: 4628 nodes, 32902 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Matched 201 proteins between data and KG.


Linking Samples: 100%|██████████| 5833/5833 [00:00<00:00, 5891.14it/s]


Saved graph to ./data/GEO/GSE33000_ad_hd/map_ad_kg/G_patient_HealthyKG_std.pkl: 4591 nodes, 18490 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Matched 201 proteins between data and KG.


Linking Samples: 100%|██████████| 32986/32986 [00:04<00:00, 6923.14it/s]


Saved graph to ./data/GEO/GSE33000_ad_hd/map_ad_kg/G_patient_HealthyKG_logfc.pkl: 4628 nodes, 72796 edges


##### (3) generate patient_2KG graphs -- merge
+ all samples link to proteins in ADKG
+ all smaples link to proteins in HealthKG
+ Merge two networks using sample nodes as bridge

In [ ]:
def merge_2kg(G, H, output_dir:str, dataset:str, scoring_method:str):
    combined = nx.compose(G, H)

    os.makedirs(output_dir, exist_ok=True)
    save_dir = os.path.join(output_dir, f"G_{dataset}_merge_{scoring_method}.pkl")
    with open(save_dir, 'wb') as f:
        pickle.dump(combined, f)

    print(f"Save graph to {save_dir}: {combined.number_of_nodes()} nodes, {combined.number_of_edges()} edges")
    return combined

##### (4) generate patient_2KG graphs
+ firstly generate a patient graph based on expression data
+ Controls link to proteins in Healthy_Aging_KG
+ Patients link to proteins in ADKG

+ Control group proteins to link: scoring functions (ecdf, logfc, std) use Disease group patients as baselines to search 'radical' points compared to Disease-Expression-Profile,
+ Disease group proteins to link: scoring functions (ecdf, logfc, std) use Control group samples as baselines to search 'radical' points compared to Control-Expression-Profile,

In [73]:
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split


def build_knn_graph_with_masks(features_df, labels_series, k=8, metric='cosine', 
                              add_label_edges=True, rewire_edges=True,
                              split_ratio=(0.7, 0.15, 0.15),
                              base_graph_type=nx.DiGraph):
    """
    Builds a k-NN graph with Patient IDs as node names, original rewiring logic,
    and randomized train/val/test masks.
    """
    # 1. Shuffle and Split Data
    patient_ids = features_df.index.tolist()
    # Shuffle indices to mix Disease and Control
    shuffled_ids = np.random.permutation(patient_ids)
    
    train_ids, temp_ids = train_test_split(shuffled_ids, train_size=split_ratio[0], stratify=labels_series.loc[shuffled_ids])
    val_size_adj = split_ratio[1] / (split_ratio[1] + split_ratio[2])
    val_ids, test_ids = train_test_split(temp_ids, train_size=val_size_adj, stratify=labels_series.loc[temp_ids])

    # Map ID to its position in the original matrix for k-NN lookup
    id_to_pos = {pid: i for i, pid in enumerate(patient_ids)}
    pos_to_id = {i: pid for i, pid in enumerate(patient_ids)}
    
    features_matrix = features_df.values
    labels_array = labels_series.values
    
    # 2. Find k-nearest neighbors
    nbrs = NearestNeighbors(n_neighbors=k, metric=metric).fit(features_matrix)
    dists, inds = nbrs.kneighbors(features_matrix)
    
    edge_list = [] # Store as (u_id, v_id, weight)
    
    # 3. Initial k-NN edges
    for i in range(len(patient_ids)):
        u = pos_to_id[i]
        for r in range(1, k):
            v_idx = inds[i, r]
            v = pos_to_id[v_idx]
            sim = 1.0 - dists[i, r]
            edge_list.append((u, v, sim))
    
    # 4. Add Label-based Edges
    if add_label_edges:
        for i, u in enumerate(tqdm(patient_ids, desc="Adding label edges")):
            same_label_idx = np.where(labels_array == labels_array[i])[0]
            knn_neighbors = inds[i, 1:]
            
            # Intersection of same label and k-NN neighbors
            valid_neighbors = [j for j in same_label_idx if j in knn_neighbors]
            for j in valid_neighbors:
                v = pos_to_id[j]
                pos_in_inds = np.where(inds[i, :] == j)[0][0]
                sim = 1.0 - dists[i, pos_in_inds]
                edge_list.append((u, v, sim))

    # 5. Original Rewire Logic
    edges_to_remove = set()
    if rewire_edges:
        rewired_count = 0
        for i, u in enumerate(tqdm(patient_ids, desc="Rewiring")):
            neighbors_idx = inds[i, 1:]
            to_remove_idx = []
            
            for j in neighbors_idx:
                sim = 1.0 - dists[i, np.where(inds[i, :] == j)[0][0]]
                if labels_array[j] != labels_array[i] and sim < 0.65:
                    to_remove_idx.append(j)
                    # Mark for global removal
                    v = pos_to_id[j]
                    edges_to_remove.add(tuple(sorted((u, v))))

            same_label_idx = [j for j in range(len(patient_ids)) if labels_array[j] == labels_array[i] and j != i and j not in neighbors_idx]
            
            if to_remove_idx and same_label_idx:
                # Find best same-label candidates to replace the removed ones
                sims = [1.0 - dists[i, np.where(inds[i, :] == j)[0][0]] if j in inds[i, :] else 0.0 for j in same_label_idx]
                top_k_indices = np.argsort(sims)[-len(to_remove_idx):]
                
                for idx in top_k_indices:
                    j = same_label_idx[idx]
                    v = pos_to_id[j]
                    if sims[idx] > 0.0:
                        edge_list.append((u, v, sims[idx]))
                        rewired_count += 1
        print(f"Rewired {rewired_count} edges.")

    # 6. Final Graph Construction
    G = base_graph_type()
    
    # Add nodes with IDs and masks
    for pid in patient_ids:
        G.add_node(pid, 
                   x=features_df.loc[pid].values, 
                   y=int(labels_series.loc[pid]),
                   train_mask=(pid in train_ids),
                   val_mask=(pid in val_ids),
                   test_mask=(pid in test_ids),
                   type='Patient') # Added type for HeteroData clarity
    
    # Add edges
    for u, v, w in edge_list:
        if tuple(sorted((u, v))) not in edges_to_remove:
            # For DiGraph, we add edges in both directions to simulate undirected similarity
            G.add_edge(u, v, weight=float(w), relation='similar')
            G.add_edge(v, u, weight=float(w), relation='similar')
            
    return G

In [74]:
class PatientNetworkGenerator:
    def __init__(self, kg_disease, kg_healthy):
        """
        Initializes with two knowledge graphs.
        :param kg_disease: KG used for Disease sample mapping (e.g., AD)
        :param kg_control: KG used for Control sample mapping
        """
        self.kg_disease = kg_disease
        self.kg_control = kg_healthy
        self.relation_map = {1: 'up_reg', -1: 'down_reg'}

    def _get_symbol_mapping(self, graph: nx.Graph, pattern: str):
        """Helper to create symbol -> node mapping for a specific graph."""
        mapping = {}
        for node in graph.nodes:
            symbol = gene_symbol_extractor(node, pattern)
            if symbol:
                mapping[symbol] = node
        return mapping

    def generate_hybrid_network(
                                self, 
                                data: pd.DataFrame, 
                                exp_df: pd.DataFrame, 
                                pattern_disease: str,
                                pattern_control:str,
                                disease_label: int = 1,
                                control_label: int = 0
                            ) -> Tuple[nx.Graph, pd.DataFrame]:
        """
        Generates a combined network:
        1. Patient-Patient network via K-NN clustering based on Cosine Similarity.
        2. Disease Patients -> KG_Disease.
        3. Control Patients -> KG_Control.
        """
        # 1. Setup and Mappings
        patient_labels = data['label'].to_dict()
        scores = data.drop(columns=['label'])
        
        map_disease = self._get_symbol_mapping(self.kg_disease, pattern_disease)
        map_control = self._get_symbol_mapping(self.kg_control, pattern_control)

        # Initialize the big network with both KGs combined
        # nx.compose merges nodes and edges from both graphs
        full_graph = nx.compose(self.kg_disease, self.kg_control)
        
        # 2. Add Patient-Patient Similarity Edges (Cosine)
        print("Contructing Patient-Patient Netwrok")
        patient_graph = build_knn_graph_with_masks(features_df=exp_df,
                                                   labels_series=data['label'],
                                                   k=5,
                                                   base_graph_type=type(full_graph))
        full_graph = nx.compose(full_graph, patient_graph)
        
        # 3. Add Patient-Protein Edges based on Label
        summary_df = pd.DataFrame(0, index=data.index, columns=['pos_edges', 'neg_edges'])
        
        # Identify radicals to iterate over
        all_common = set(map_disease.keys()) | set(map_control.keys())
        common_cols = [c for c in scores.columns if c in all_common and c in exp_df.columns]
        radicals = scores[common_cols].stack()
        radicals = radicals[radicals != 0]

        for (patient, symbol), val in tqdm(radicals.items(), total=len(radicals), desc="Linking Samples to KGs"):
            label = patient_labels[patient]
            
            # Determine which KG to use
            if label == disease_label and symbol in map_disease:
                target_node = map_disease[symbol]
            elif label == control_label and symbol in map_control:
                target_node = map_control[symbol]
            else:
                continue # Skip if gene isn't in the respective KG for that label

            rel = self.relation_map.get(int(val))
            weight = float(exp_df.loc[patient, symbol])

            # Add forward and reverse edges
            full_graph.add_edge(patient, target_node, relation=rel, weight=weight)
            full_graph.add_edge(target_node, patient, relation=f'rev_{rel}', weight=weight)
            
            col = 'pos_edges' if int(val) == 1 else 'neg_edges'
            summary_df.at[patient, col] += 1

        return full_graph, summary_df

In [75]:
def generat_and_save_hybrid(exp_path:str, 
                     scoring_path:str, 
                     kg_disease_path:str, 
                     kg_health_path:str,
                     pattern_disease:str,
                     pattern_control:str,
                     output_dir:str,
                     scoring_method:str='ecdf',
                     dataset:str = 'adni'
                     ):

    # 1. Load expression df, smaple scoring df, KG
    exp_df = pd.read_csv(exp_path, index_col=0)
    if exp_df.shape[0] > exp_df.shape[1]:
        exp_df = exp_df.transpose()
    data = pd.read_csv(scoring_path, index_col=0)
    kg_disease = load_graph(kg_disease_path)
    kg_control = load_graph(kg_health_path)

    # clean exp_df before K-NN
    # drop genes with no variation
    exp_df = exp_df.loc[:, exp_df.std() > 0]
    # Using median is usually safer for gene expression
    exp_df = exp_df.fillna(exp_df.median())
    # normalize safely
    min_val = exp_df.min()
    max_val = exp_df.max()
    exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
    # final fill-na
    exp_norm = exp_norm.fillna(0)

    # 2. Generate PatientNetwork
    png = PatientNetworkGenerator(kg_disease=kg_disease,
                                  kg_healthy=kg_control)
    network, summary = png.generate_hybrid_network(data=data,
                                    exp_df=exp_df,
                                    pattern_disease=pattern_disease,
                                    pattern_control=pattern_control,
                                    disease_label=1,
                                    control_label=0
                                    )
    
    # 3. save
    os.makedirs(output_dir, exist_ok=True)
    save_network = os.path.join(output_dir, f"G_{dataset}_hybrid_{scoring_method}.pkl")
    save_graph(network, save_network)

    save_summary = os.path.join(output_dir, f"Summary_{dataset}_hybrid_{scoring_method}.csv")
    summary.to_csv(save_summary)

    return network, summary

ADNI hybrid graph

In [80]:
adni_hybrid_ecdf = generat_and_save_hybrid(
                                            exp_path="./data/ADNI/adni_exp_2cls.csv",
                                            scoring_path="./data/ADNI/map_ad_kg/sample_scoring_ecdf.csv",
                                            kg_disease_path="./data/KG/ad_network_with_reverse_edges.pkl",
                                            kg_health_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern_disease=pattern_hgnc,
                                            pattern_control=pattern_uniprotkb,
                                            output_dir="./data/ADNI/",
                                            scoring_method='ecdf',
                                            dataset='adni')
adni_hybrid_std = generat_and_save_hybrid(
                                            exp_path="./data/ADNI/adni_exp_2cls.csv",
                                            scoring_path="./data/ADNI/map_ad_kg/sample_scoring_std.csv",
                                            kg_disease_path="./data/KG/ad_network_with_reverse_edges.pkl",
                                            kg_health_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern_disease=pattern_hgnc,
                                            pattern_control=pattern_uniprotkb,
                                            output_dir="./data/ADNI/",
                                            scoring_method='std',
                                            dataset='adni')

Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Contructing Patient-Patient Netwrok


Rewiring: 100%|██████████| 455/455 [00:01<00:00, 427.08it/s]


Rewired 0 edges.


Linking Samples to KGs: 100%|██████████| 38749/38749 [00:03<00:00, 11887.05it/s]


Saved graph to ./data/ADNI/G_adni_hybrid_ecdf.pkl: 8275 nodes, 72920 edges
Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Contructing Patient-Patient Netwrok


Rewiring: 100%|██████████| 455/455 [00:00<00:00, 754.45it/s]


Rewired 0 edges.


Linking Samples to KGs: 100%|██████████| 46319/46319 [00:05<00:00, 9257.74it/s] 


Saved graph to ./data/ADNI/G_adni_hybrid_std.pkl: 8275 nodes, 81550 edges


GEO hybrid graph

In [77]:
geo_hybrid_ecdf = generat_and_save_hybrid(
                                            exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_ad_kg/sample_scoring_ecdf.csv",
                                            kg_disease_path="./data/KG/ad_network_with_reverse_edges.pkl",
                                            kg_health_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern_disease=pattern_hgnc,
                                            pattern_control=pattern_uniprotkb,
                                            output_dir="./data/GEO/GSE33000_ad_hd/",
                                            scoring_method='ecdf',
                                            dataset='geo')

Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Contructing Patient-Patient Netwrok


Rewiring: 100%|██████████| 467/467 [00:00<00:00, 589.58it/s]


Rewired 0 edges.


Linking Samples to KGs: 100%|██████████| 80295/80295 [00:11<00:00, 6750.58it/s] 


Saved graph to ./data/GEO/GSE33000_ad_hd/G_geo_hybrid_ecdf.pkl: 8287 nodes, 158278 edges


In [79]:
geo_hybrid_logfc = generat_and_save_hybrid(
                                            exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_ad_kg/sample_scoring_logfc.csv",
                                            kg_disease_path="./data/KG/ad_network_with_reverse_edges.pkl",
                                            kg_health_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern_disease=pattern_hgnc,
                                            pattern_control=pattern_uniprotkb,
                                            output_dir="./data/GEO/GSE33000_ad_hd/",
                                            scoring_method='logfc',
                                            dataset='geo')

Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges
Contructing Patient-Patient Netwrok


Rewiring: 100%|██████████| 467/467 [00:01<00:00, 441.59it/s]


Rewired 0 edges.


Linking Samples to KGs: 100%|██████████| 132658/132658 [00:17<00:00, 7741.95it/s] 


Saved graph to ./data/GEO/GSE33000_ad_hd/G_geo_hybrid_logfc.pkl: 8287 nodes, 228134 edges


In [ ]:
geo_hybrid_std = generat_and_save_hybrid(
                                            exp_path="./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv",
                                            scoring_path="./data/GEO/GSE33000_ad_hd/map_ad_kg/sample_scoring_std.csv",
                                            kg_disease_path="./data/KG/ad_network_with_reverse_edges.pkl",
                                            kg_health_path='./data/KG/healthy_aging_kg.pkl',
                                            pattern_disease=pattern_hgnc,
                                            pattern_control=pattern_uniprotkb,
                                            output_dir="./data/GEO/GSE33000_ad_hd/",
                                            scoring_method='std',
                                            dataset='geo')

KG Visualization

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def visualize_hybrid_graph(G, title="Hybrid Patient-KG Network"):
    """
    Visualizes the hybrid graph with color-coded nodes.
    - Patient Nodes: Red
    - Disease KG Nodes: Blue
    - Control KG Nodes: Green
    - Overlapping KG Nodes: Cyan
    """
    plt.figure(figsize=(12, 12))
    
    # 1. Identify Node Categories
    # We use the 'type' attribute we set earlier, or check which KG they came from
    node_colors = []
    for node, data in G.nodes(data=True):
        node_type = data.get('type', 'Unknown')
        
        if node_type == 'Patient':
            node_colors.append('red')
        elif 'UniProtKB' in str(node) or 'UniProtKB' in str(data): # Logic for your specific KG node naming
            # If a node exists in both KGs, you might want a special color
            node_colors.append('skyblue')
        elif 'HGNC' in str(node) or 'HGNC' in str(data):
            node_colors.append('lightgreen')
        else:
            node_colors.append('gray')

    # 2. Layout Calculation
    # 'sfdp' or 'neato' are better for large graphs, but 'spring' is built-in
    print("Calculating layout (this may take a moment for large KGs)...")
    pos = nx.spring_layout(G, k=0.15, iterations=20)

    # 3. Draw Nodes
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=30, alpha=0.8)

    # 4. Draw Edges with transparency to see density
    # Patient-Patient edges can be highlighted or made thinner
    nx.draw_networkx_edges(G, pos, alpha=0.1, edge_color='gray', width=0.5)

    plt.title(title)
    plt.axis('off')
    
    # Custom Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Patients', markerfacecolor='red', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Disease KG', markerfacecolor='skyblue', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Control KG', markerfacecolor='lightgreen', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.show()

In [ ]:
visualize_hybrid_graph(geo_hybrid_logfc[0])